# SRQ generalization M11 — adaptive precision (train-only)

This source-locked follow-up tests one fixed 25% FP16 allowance at widths 10k and 20k. Upload the exact M6 artifact when requested. Run every cell in order; the notebook never materializes CIFAR-100 test features.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m11_cifar_features'
OUTPUT_DIR='/content/srq_m11_adaptive_output'
EXPECTED_M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
EXPECTED_M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m11_adaptive_precision_train_only.json':'1fc5e740b2236a52943d79485c737508f0684366bdf53a4aede267971c3b30d9',
 'tools/srq_generalization_m11.py':'b13f0ad5ed81c33a61ecca53ff536bed3f7dd8c8b0731ef4cfe5a1a8adfe0651',
 'methods/analytic_ridge/adaptive_upper.py':'d34eae6072ea3e8736b6fb940c7a0691c845d8217569c7b6fe33f4b9e143c5dc',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/analytic_ridge/__init__.py':'59babe9c4881a0991a7f5825958cd0ef8c41f4d6b88b1b7cfc843b6e9ba99aed',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m11_adaptive_precision_train_only.json'
RUNNER='tools/srq_generalization_m11.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('M11 SOURCE LOCK: PASS')

In [ ]:
# Local correctness gates before data download.
command=[sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_adaptive_analytic_ridge.py','tests/test_srq_generalization_m11.py','tests/test_analytic_ridge_backend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M11 local gates failed; return the complete traceback.'
print('M11 LOCAL GATES: PASS')

In [ ]:
# Upload the exact M6 artifact used to lock reference results and identities.
from google.colab import files
uploaded=files.upload()
assert EXPECTED_M6_NAME in uploaded,f'Upload {EXPECTED_M6_NAME} exactly.'
uploaded_path=Path(EXPECTED_M6_NAME).resolve()
source_path=Path('/content')/EXPECTED_M6_NAME
if source_path.exists(): source_path.unlink()
shutil.move(str(uploaded_path),str(source_path))
assert sha_raw(source_path)==EXPECTED_M6_SHA,(sha_raw(source_path),EXPECTED_M6_SHA)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Artifact upload contaminated the repository.'
SOURCE_M6_ARTIFACT=str(source_path)
print('M6 ARTIFACT LOCK: PASS',SOURCE_M6_ARTIFACT)

In [ ]:
# Download locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha_raw(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m11','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run the single preregistered adaptive policy at widths 10k and 20k.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--source-m6-artifact',SOURCE_M6_ARTIFACT,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M11 START: adaptive INT8/FP16, fixed 25% byte interval, widths 10k/20k.',flush=True)
completed=subprocess.run(command)
RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m11_results.json'
assert result_path.is_file(),'M11 failed before writing a result; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Compact comparison table against source-locked M6 references.
import pandas as pd
rows=[]
for item in result['width_results']:
    ref=item['source_m6_reference']
    rows.append({'width':item['width'],'Exact AIA':ref['exact_validation_aia_percent'],'P2B AIA':ref['p2b_validation_aia_percent'],'Adaptive AIA':item['validation_aia_percent'],'Adaptive loss':item['adaptive_validation_aia_loss_pp'],'Adaptive-P2B':item['adaptive_minus_p2b_validation_aia_pp'],'Adaptive state MiB':item['final_total_persistent_bytes']/2**20,'state reduction':item['adaptive_total_state_reduction_fraction']})
display(pd.DataFrame(rows))

In [ ]:
# Source-derived vector figures.
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(10.5,4.0))
for item in result['width_results']:
    tasks=[r['task'] for r in item['records']]
    axes[0].plot(tasks,[r['adaptive_relative_factor_error'] for r in item['records']],marker='o',label=f"Adaptive {item['width']//1000}k")
    axes[0].plot(tasks,[r['all_int8_relative_factor_error'] for r in item['records']],linestyle='--',label=f"INT8 proxy {item['width']//1000}k")
widths=[item['width'] for item in result['width_results']]
axes[1].plot(widths,[item['source_m6_reference']['exact_validation_aia_percent'] for item in result['width_results']],marker='o',label='Exact M6')
axes[1].plot(widths,[item['source_m6_reference']['p2b_validation_aia_percent'] for item in result['width_results']],marker='o',label='P2B M6')
axes[1].plot(widths,[item['validation_aia_percent'] for item in result['width_results']],marker='o',label='Adaptive M11')
axes[0].set_xlabel('Task'); axes[0].set_ylabel('Local factor relative error'); axes[0].set_yscale('log')
axes[1].set_xlabel('Random-feature width'); axes[1].set_ylabel('Validation AIA (%)')
for ax in axes: ax.grid(True,alpha=.25); ax.legend(fontsize=8)
fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/'m11_adaptive_precision.svg'
fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file()

In [ ]:
# Export evidence whether PASS or FAIL; caches/checkpoints are excluded.
bundle=Path('/content/srq_generalization_m11_adaptive_precision_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for source,name in [(Path(OUTPUT_DIR)/'m11_results.json','m11_results.json'),(Path(OUTPUT_DIR)/'m11_task_trajectory.csv','m11_task_trajectory.csv'),(Path(OUTPUT_DIR)/'m11_adaptive_precision.svg','m11_adaptive_precision.svg'),(Path(CONFIG),'config.json'),(Path('docs/research/SRQ_GENERALIZATION_M11_RUNBOOK.md'),'runbook.md')]: shutil.copy2(source,bundle/name)
manifest={'schema_version':1,'status':result['status'],'uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_m6_sha256':EXPECTED_M6_SHA,'files':{p.name:sha_raw(p) for p in bundle.iterdir()}}
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha_raw(archive))
files.download(archive)
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M11_ADAPTIVE_PRECISION_TRAIN_ONLY','M11 returned FAIL; keep the downloaded artifact and do not relax or retry the policy.'